<a href="https://colab.research.google.com/github/selim679/Backblaze-panne-disques-durs/blob/main/notebookbacklaze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

salimfekihsalem_backblaze1_path = kagglehub.dataset_download('salimfekihsalem/backblaze1')
salimfekihsalem_backblaze_pkl1_path = kagglehub.dataset_download('salimfekihsalem/backblaze-pkl1')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import pandas as pd
import numpy as np
import os, gc
from tqdm import tqdm

MODEL    = 'ST4000DM000'
DATA_DIR = '/kaggle/working/data/'
os.makedirs(DATA_DIR, exist_ok=True)

PKL_TRAIN = DATA_DIR + 'Lab1-2017-Full-ST4000DM000.pkl'
PKL_TEST  = DATA_DIR + 'Lab1-2016-Q4-ST4000DM000.pkl'

folders_2017 = [
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q1_2017',
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q2_2017',
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q3_2017',
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q4_2017',
]
folders_2016_q4 = [
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q4_2016',
]

def load_filtered_and_save(folders, pkl_path, model=MODEL):
    """Charge CSV un par un, filtre sur le modèle, sauvegarde en PKL."""

    # Colonnes à garder (métadonnées + SMART brutes uniquement)
    META = ['date', 'serial_number', 'model', 'capacity_bytes', 'failure']

    chunks = []
    total_rows = 0

    for folder in folders:
        if not os.path.exists(folder):
            print(f"⚠️  Manquant : {folder}")
            continue

        files = sorted([
            os.path.join(folder, f)
            for f in os.listdir(folder) if f.endswith('.csv')
        ])
        print(f"\n📂 {os.path.basename(folder)} → {len(files)} fichiers")

        for file in tqdm(files):
            try:
                # 1. Lire seulement les colonnes nécessaires
                # D'abord récupère les colonnes disponibles
                cols_available = pd.read_csv(file, nrows=0).columns.tolist()

                # Garde META + toutes les colonnes smart_X_raw (pas normalized)
                smart_raw = [c for c in cols_available
                             if 'smart_' in c and '_normalized' not in c]
                cols_to_read = META + smart_raw
                cols_to_read = [c for c in cols_to_read if c in cols_available]

                # 2. Lire avec filtre colonnes
                tmp = pd.read_csv(file, usecols=cols_to_read, low_memory=False)

                # 3. Filtrer sur le modèle
                tmp = tmp[tmp['model'] == model].copy()

                if tmp.empty:
                    continue

                # 4. Réduire la mémoire immédiatement
                for col in tmp.select_dtypes('float64').columns:
                    tmp[col] = tmp[col].astype('float32')
                for col in tmp.select_dtypes('int64').columns:
                    tmp[col] = tmp[col].astype('int32')

                total_rows += len(tmp)
                chunks.append(tmp)
                del tmp

            except Exception as e:
                print(f"❌ {file}: {e}")

        # Libérer après chaque quarter
        gc.collect()

    if not chunks:
        raise ValueError("Aucune donnée !")

    print(f"\n🔗 Concaténation de {len(chunks)} chunks ({total_rows} lignes)...")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    ram = df.memory_usage(deep=True).sum() / 1e9
    print(f"✅ Shape : {df.shape}  |  RAM : {ram:.2f} GB")
    print(f"   Pannes : {df['failure'].sum()}")

    print(f"💾 Sauvegarde → {pkl_path}")
    df.to_pickle(pkl_path)
    print("✅ PKL sauvegardé !")

    return df

# --- Génération TRAIN ---
print("=" * 50)
print("GÉNÉRATION TRAIN (2017)")
print("=" * 50)
df = load_filtered_and_save(folders_2017, PKL_TRAIN)

# --- Génération TEST ---
print("\n" + "=" * 50)
print("GÉNÉRATION TEST (2016 Q4)")
print("=" * 50)
df_t = load_filtered_and_save(folders_2016_q4, PKL_TEST)

print("\n✅ TERMINÉ")
print(f"Fichiers générés :")
print(f"  {PKL_TRAIN}  ({os.path.getsize(PKL_TRAIN)/1e6:.1f} MB)")
print(f"  {PKL_TEST}   ({os.path.getsize(PKL_TEST)/1e6:.1f} MB)")

In [ ]:
# ============================================================
# CHARGEMENT RAPIDE (après upload des PKL)
# ============================================================
import pandas as pd

df   = pd.read_pickle('/kaggle/input/datasets/salimfekihsalem/backblaze-pkl1/Lab1-2017-Full-ST4000DM000.pkl')
df_t = pd.read_pickle('/kaggle/input/datasets/salimfekihsalem/backblaze-pkl1/Lab1-2016-Q4-ST4000DM000.pkl')

print("TRAIN :", df.shape,   "| Pannes :", df['failure'].sum())
print("TEST  :", df_t.shape, "| Pannes :", df_t['failure'].sum())

In [ ]:
df.head()

In [ ]:

def analyze_columns(df, keyword='smart'):
    """
    Analyse toutes les colonnes contenant un mot-clé.
    Retourne un DataFrame avec les statistiques de chaque colonne.
    """
    # Sélectionner uniquement les colonnes contenant le mot-clé
    smart_cols = [c for c in df.columns if keyword in c.lower()]

    results = []
    for col in smart_cols:
        n_unique = df[col].nunique(dropna=True)
        col_min  = df[col].min(skipna=True)
        col_max  = df[col].max(skipna=True)
        n_nan    = df[col].isna().sum()

        is_constant = (n_unique <= 1) or (col_min == col_max)

        results.append({
            'colonne'      : col,
            'n_unique'     : n_unique,
            'min'          : col_min,
            'max'          : col_max,
            'n_nan'        : n_nan,
            'est_constante': is_constant
        })

    return pd.DataFrame(results)

# --- Analyse ---
print("Analyse des colonnes SMART...")
col_stats = analyze_columns(df, keyword='smart')

# Listes constantes vs informatives
const_cols = col_stats[col_stats['est_constante']]['colonne'].tolist()
info_cols  = col_stats[~col_stats['est_constante']]['colonne'].tolist()

print(f"\n📊 Colonnes SMART totales    : {len(col_stats)}")
print(f"   Colonnes CONSTANTES       : {len(const_cols)}")
print(f"   Colonnes INFORMATIVES     : {len(info_cols)}")
print(f"\nColonnes constantes : {const_cols}")
print(f"\nColonnes informatives : {info_cols}")

In [ ]:
analyze_columns(df, keyword='smart')

In [ ]:
analyze_columns(df, keyword='')

In [ ]:

print(f"Dimensions AVANT suppression : {df.shape}")

df   = df.drop(columns=const_cols, errors='ignore')
df_t = df_t.drop(columns=const_cols, errors='ignore')

print(f"Dimensions APRÈS suppression : {df.shape}")
print(f"Colonnes supprimées          : {len(const_cols)}")

# Vérification : aucune variabilité dans les colonnes supprimées
print("\n Vérification des colonnes supprimées (toutes doivent être constantes) :")
print(col_stats[col_stats['est_constante']][['colonne','n_unique','min','max']].to_string(index=False))

In [ ]:
# ============================================================
# SECTION 2.3 — Nettoyage et sélection finale
# ============================================================

import gc

# ---- ÉTAPE 1 : Supprimer colonnes entièrement vides --------
before = df.shape[1]
df   = df.dropna(axis=1, how='all')
df_t = df_t.dropna(axis=1, how='all')
print(f"Colonnes entièrement vides supprimées : {before - df.shape[1]}")

# ---- ÉTAPE 2 : Supprimer lignes avec valeurs critiques manquantes ----
critical_cols = ['date', 'serial_number', 'failure']
before = len(df)
df   = df.dropna(subset=critical_cols)
df_t = df_t.dropna(subset=critical_cols)
print(f"Lignes supprimées (valeurs critiques manquantes) : {before - len(df)}")

# ---- ÉTAPE 3 : Colonnes avec valeurs manquantes restantes ----
nan_cols = df.columns[df.isna().any()].tolist()
print(f"\nColonnes avec NaN restants ({len(nan_cols)}) :")
print(df[nan_cols].isna().sum().sort_values(ascending=False).head(10))

# ---- ÉTAPE 4 : Remplacement des NaN par 0 ----
# Justification : les attributs SMART manquants signifient
# souvent que le disque ne supporte pas cet attribut → valeur neutre = 0
df[nan_cols]   = df[nan_cols].fillna(0)
df_t[nan_cols] = df_t[nan_cols].fillna(0)
print(f"\nNaN restants après remplacement : {df.isna().sum().sum()}")

# ---- ÉTAPE 5 : Séparation normaux / défaillants ----
df_failed  = df[df['failure'] == 1].copy()
df_normal  = df[df['failure'] == 0].copy()
print(f"\nDisques défaillants (failure=1) : {len(df_failed)} lignes")
print(f"Disques normaux    (failure=0) : {len(df_normal)} lignes")

In [ ]:
# ---- ÉTAPE 6 & 7 : Sélection des variables SMART informatives ----
META_COLS = ['date', 'serial_number', 'failure']

# Variables brutes uniquement (pas normalized, déjà supprimées avant)
smart_raw_cols = [c for c in df.columns
                  if 'smart_' in c and '_normalized' not in c]

# Liste finale des colonnes
final_cols = META_COLS + smart_raw_cols

print(f"Colonnes métadonnées  : {len(META_COLS)}")
print(f"Colonnes SMART brutes : {len(smart_raw_cols)}")
print(f"Total colonnes finales: {len(final_cols)}")
print(f"\nColonnes SMART conservées : {smart_raw_cols}")

In [ ]:
# ---- ÉTAPE 8 & 9 : Appliquer les mêmes colonnes au test ----
# Garder seulement les colonnes finales
df   = df[[c for c in final_cols if c in df.columns]].copy()
df_t = df_t[[c for c in final_cols if c in df_t.columns]].copy()

# Vérification stricte
assert list(df.columns) == list(df_t.columns), "❌ Colonnes différentes entre train et test !"
print("✅ Colonnes identiques entre train et test")
print(f"   Ordre identique : {list(df.columns) == list(df_t.columns)}")

# ---- ÉTAPE 10 : Vérification finale ----
print(f"\n=== VÉRIFICATION FINALE ===")
print(f"TRAIN : {df.shape}  | NaN : {df.isna().sum().sum()} | Types : {df.dtypes.value_counts().to_dict()}")
print(f"TEST  : {df_t.shape} | NaN : {df_t.isna().sum().sum()} | Types : {df_t.dtypes.value_counts().to_dict()}")
display(df.head(3))

In [ ]:
# ============================================================
# SECTION 2.4 — Sauvegarde en Pickle
# ============================================================

import os, time, gc

DATA_DIR = '/kaggle/working/data/'
os.makedirs(DATA_DIR, exist_ok=True)  # 🔥 FIX

PKL_TRAIN = DATA_DIR + 'Lab1-2017-Full-ST4000DM000.pkl'
PKL_TEST  = DATA_DIR + 'Lab1-2016-Q4-ST4000DM000.pkl'

print("💾 Sauvegarde TRAIN...")
df.to_pickle(PKL_TRAIN)

print("💾 Sauvegarde TEST...")
df_t.to_pickle(PKL_TEST)

# Vérification
df_check = pd.read_pickle(PKL_TRAIN)
print(f"\n✅ PKL rechargé : {df_check.shape} == {df.shape} → {df_check.shape == df.shape}")

# Temps de chargement
t0 = time.time()
pd.read_pickle(PKL_TRAIN)
print(f"⚡ Chargement PKL  : {time.time()-t0:.3f}s")

# Taille
print(f"\nTaille fichiers :")
print(f"  Train PKL : {os.path.getsize(PKL_TRAIN)/1e6:.1f} MB")
print(f"  Test  PKL : {os.path.getsize(PKL_TEST)/1e6:.1f} MB")

del df_check
gc.collect()